In [ ]:
import os
import numpy as np
import pandas as pd
import joblib
import mlflow
import mlflow.lightgbm
import warnings
warnings.filterwarnings('ignore')

from lightgbm import LGBMClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score,
    recall_score, roc_auc_score,
    classification_report,
)
from sklearn.utils.class_weight import compute_class_weight

os.chdir(r'D:\End-to-end-ML\Customer-Risk-Escalation-Engine')

mlflow.set_tracking_uri(
    'file:///D:/End-to-end-ML/Customer-Risk-Escalation-Engine/mlruns'
)


d:\End-to-end-ML\Customer-Risk-Escalation-Engine\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
X_train = pd.read_csv('data/processed_data/X_train.csv')
X_test  = pd.read_csv('data/processed_data/X_test.csv')
y_train = pd.read_csv('data/processed_data/y_train.csv').squeeze()
y_test  = pd.read_csv('data/processed_data/y_test.csv').squeeze()

train_embeddings = np.load('data/processed_data/train_embeddings.npy')
test_embeddings  = np.load('data/processed_data/test_embeddings.npy')

print("Tabular splits:")
print(f"  X_train : {X_train.shape}")
print(f"  X_test  : {X_test.shape}")
print(f"  y_train : {y_train.shape}")
print(f"  y_test  : {y_test.shape}")

print("\nEmbeddings:")
print(f"  train_embeddings : {train_embeddings.shape}")
print(f"  test_embeddings  : {test_embeddings.shape}")

Tabular splits:
  X_train : (80000, 12)
  X_test  : (20000, 12)
  y_train : (80000,)
  y_test  : (20000,)

Embeddings:
  train_embeddings : (80000, 768)
  test_embeddings  : (20000, 768)


In [3]:
assert X_train.shape[0] == train_embeddings.shape[0], \
    f"MISMATCH: X_train {X_train.shape[0]} vs embeddings {train_embeddings.shape[0]}"

assert X_test.shape[0] == test_embeddings.shape[0], \
    f"MISMATCH: X_test {X_test.shape[0]} vs embeddings {test_embeddings.shape[0]}"

print(f"Train alignment: {X_train.shape[0]:,} rows match")
print(f"Test alignment: {X_test.shape[0]:,} rows match")

Train alignment: 80,000 rows match
Test alignment: 20,000 rows match


In [ ]:
#Converting tabular to numpy
X_train_tab = X_train.values
X_test_tab  = X_test.values

#Concatenated along feature axis
X_train_fused = np.concatenate([X_train_tab, train_embeddings], axis=1)
X_test_fused  = np.concatenate([X_test_tab,  test_embeddings],  axis=1)

print(f"  Tabular features   : {X_train_tab.shape[1]}")
print(f"  Embedding features : {train_embeddings.shape[1]}")
print(f"  Combined features  : {X_train_fused.shape[1]}")
print(f"\n  X_train_fused : {X_train_fused.shape}")
print(f"  X_test_fused  : {X_test_fused.shape}")

  Tabular features   : 12
  Embedding features : 768
  Combined features  : 780

  X_train_fused : (80000, 780)
  X_test_fused  : (20000, 780)


In [ ]:
classes = np.array([0, 1])
weights = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=y_train # type: ignore
)
class_weight_dict = dict(zip(classes, weights))

print(f"Class weights : {class_weight_dict}")
print(f"Imbalance ratio : {(y_train==0).sum() / (y_train==1).sum():.1f}:1") # type: ignore

#formula to calculate weights : -
# weight for class  = total_samples / (number_of_classes × count_of_class)

''' 
weight for class 0 = 1000 / (2 × 900) = 1000 / 1800 = 0.556
weight for class 1 = 1000 / (2 × 100) = 1000 / 200  = 5.0

'''

Class weights : {0: 0.5560652820641143, 1: 4.959087527894868}
Imbalance ratio : 8.9:1


' \nweight for class 0 = 1000 / (2 × 900) = 1000 / 1800 = 0.556\nweight for class 1 = 1000 / (2 × 100) = 1000 / 200  = 5.0\n\n'

In [ ]:
fusion_params = {
    "n_estimators"  : 300,
    "max_depth"     : 5,
    "learning_rate" : 0.05,
    "subsample"     : 0.8,
    "class_weight"  : class_weight_dict,
    "random_state"  : 42,
    "n_jobs"        : -1,
    "verbose"       : -1
}

mlflow.set_experiment("customer_escalation_fusion")

with mlflow.start_run(run_name="LateFusion_LightGBM"):

    #Train 
    fusion_model = LGBMClassifier(**fusion_params)
    fusion_model.fit(
        X_train_fused, y_train,
        eval_set=[(X_test_fused, y_test)]
    )

    #Predict
    y_pred      = fusion_model.predict(X_test_fused)
    y_pred_prob = fusion_model.predict_proba(X_test_fused)[:, 1] # type: ignore

    #Metrics
    metrics = {
        "accuracy"  : round(accuracy_score(y_test, y_pred), 4), # type: ignore
        "f1_score"  : round(f1_score(y_test, y_pred), 4), # type: ignore
        "precision" : round(precision_score(y_test, y_pred, zero_division=0), 4), # type: ignore
        "recall"    : round(recall_score(y_test, y_pred), 4), # type: ignore
        "roc_auc"   : round(roc_auc_score(y_test, y_pred_prob), 4) # type: ignore
    }

    #Log to MLflow
    mlflow.log_params(fusion_params)
    mlflow.log_metrics(metrics)
    mlflow.lightgbm.log_model(fusion_model, "fusion_lgbm") # type: ignore

    print("="*55)
    print("  Late Fusion Model — Results")
    print("="*55)
    print(f"  Accuracy  : {metrics['accuracy']}")
    print(f"  F1 Score  : {metrics['f1_score']}")
    print(f"  Precision : {metrics['precision']}")
    print(f"  Recall    : {metrics['recall']}")
    print(f"  ROC AUC   : {metrics['roc_auc']}")
    print("="*55)

    print("\nClassification Report:")
    print(classification_report(
        y_test, y_pred, # type: ignore
        target_names=['Not Escalated', 'Escalated'],
        zero_division=0
    ))

    print("MLflow run logged")

Traceback (most recent call last):
  File "d:\End-to-end-ML\Customer-Risk-Escalation-Engine\venv\Lib\site-packages\mlflow\store\tracking\file_store.py", line 383, in search_experiments
    exp = self._get_experiment(exp_id, view_type)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\End-to-end-ML\Customer-Risk-Escalation-Engine\venv\Lib\site-packages\mlflow\store\tracking\file_store.py", line 481, in _get_experiment
    meta = FileStore._read_yaml(experiment_dir, FileStore.META_DATA_FILE_NAME)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\End-to-end-ML\Customer-Risk-Escalation-Engine\venv\Lib\site-packages\mlflow\store\tracking\file_store.py", line 1670, in _read_yaml
    return _read_helper(root, file_name, attempts_remaining=retries)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\End-to-end-ML\Customer-Risk-Escalation-Engine\venv\Lib\site-packages\mlflow\store\tracking\file_store.py", line 1663, 

2026/07/06 22:54:56 INFO mlflow.tracking.fluent: Experiment with name 'customer_escalation_fusion' does not exist. Creating a new experiment.
Traceback (most recent call last):
  File "d:\End-to-end-ML\Customer-Risk-Escalation-Engine\venv\Lib\site-packages\mlflow\store\tracking\file_store.py", line 383, in search_experiments
    exp = self._get_experiment(exp_id, view_type)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\End-to-end-ML\Customer-Risk-Escalation-Engine\venv\Lib\site-packages\mlflow\store\tracking\file_store.py", line 481, in _get_experiment
    meta = FileStore._read_yaml(experiment_dir, FileStore.META_DATA_FILE_NAME)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\End-to-end-ML\Customer-Risk-Escalation-Engine\venv\Lib\site-packages\mlflow\store\tracking\file_store.py", line 1670, in _read_yaml
    return _read_helper(root, file_name, attempts_remaining=retries)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

  Late Fusion Model — Results
  Accuracy  : 0.7241
  F1 Score  : 0.3734
  Precision : 0.2422
  Recall    : 0.8151
  ROC AUC   : 0.8388

Classification Report:
               precision    recall  f1-score   support

Not Escalated       0.97      0.71      0.82     17983
    Escalated       0.24      0.82      0.37      2017

     accuracy                           0.72     20000
    macro avg       0.61      0.76      0.60     20000
 weighted avg       0.90      0.72      0.78     20000

MLflow run logged


In [9]:
os.makedirs('models/saved_models', exist_ok=True)
joblib.dump(fusion_model, 'models/saved_models/fusion_model.pkl')

['models/saved_models/fusion_model.pkl']